Run the following before cell excecution: 

-`!pip install ollama`
- `ollama serve`
- `ollama pull gemma3:4b` 

In [1]:
!pip install ollama

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
anaconda-cloud-auth 0.1.4 requires pydantic<2.0, but you have pydantic 2.12.5 which is incompatible.



     ---------------------------------------- 0.0/90.6 kB ? eta -:--:--
     --------------------------- ------------ 61.4/90.6 kB 1.7 MB/s eta 0:00:01
     ---------------------------------------- 90.6/90.6 kB 1.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/73.5 kB ? eta -:--:--
   ---------------------------------------- 73.5/73.5 kB 2.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/78.8 kB ? eta -:--:--
   ---------------------------------------- 78.8/78.8 kB 4.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/463.6 kB ? eta -:--:--
   --------------- ------------------------ 174.1/463.6 kB 5.3 MB/s eta 0:00:01
   ---------------------------------------  460.8/463.6 kB 5.8 MB/s eta 0:00:01
   ---------------------------------------- 463.6/463.6 kB 4.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ------- -------------------------------- 0.4/2.0 MB 8.3 MB/s eta 0:00:01
   ------------

# Raw completion template

In [ ]:
import ollama


DEFAULT_MODEL = "gemma3:4b"


# Response generation function
def complete_document_sdk(prefix: str, *, model: str = DEFAULT_MODEL, num_predict: int = 180, temperature: float = 0.5) -> str:
    response = ollama.generate(
        model=model,
        prompt=prefix,
        stream=False,
        raw=True,
        options={
            "num_predict": num_predict,
            "temperature": temperature,
        },
    )
    return response["response"]

# Conversation management
- Prompt assembly with system prompt, instruction and context (RAG - Not implemented yet)
- Response-format 
- History handling

In [33]:
class ConversationManager:
    def __init__(self, system_prompt, document_type, max_history_turns):

        self.history = []
        self.system_prompt = system_prompt
        self.document_type = document_type
        self.max_history_turns = max_history_turns  #Added a sliding window on the chat history to reduce token usage
    
    def add_user_message(self, message):
        self.history.append({"role": "user", "content": message})
        self._apply_sliding_window()
    
    def add_assistant_message(self, message):
        self.history.append({"role": "assistant", "content": message})
        self._apply_sliding_window()

    #Keep most recent N conversation turns
    def _apply_sliding_window(self): 
        max_messages = self.max_history_turns * 2
        if len(self.history) > max_messages:
            self.history = self.history[-max_messages:]
    
    #formats the chat history to a student and assistant roles for the prompt
    def _get_history_string(self):
        if not self.history:
            return ""
        parts = []
        for msg in self.history:
            role = "Student" if msg["role"] == "user" else "Assistant"
            parts.append(f"{role}: {msg['content']}")
        return "\n".join(parts)
    
    #Assembling the prompt based on document type
    def assemble_prompt(self, context, question): 
        history = self._get_history_string()
        
        #Conversational format including transition to final answer
        if self.document_type == "conversation":
            #System prompt is the introduction
            prompt = f"""{self.system_prompt} 
{history}

Student: Here is the context:
{context}

Student: {question}
Assistant: Based on the context above, my answer is:"""
        

        #Markdown format including transition to final answer
        elif self.document_type == "markdown":
            prompt = f"""# HKBU Research Assistant

## System
{self.system_prompt}

## Previous Conversation
{history}

## Context
{context}

## Question
{question}

## Answer
Based on the context above, my answer is:
"""
        #Structured document format including transition to final answer
        else:  
            prompt = f"""<prompt>
  <system>{self.system_prompt}</system>
  <history>{history}</history>
  <context>{context}</context>
  <question>{question}</question>
</prompt>

<response>
Based on the context above, my answer is:"""
        
        return prompt
    

    #Retrieving and fomatting complete_document_sdk output
    def get_response(self, user_message, context, model, temperature=0.3, max_tokens=300):
        import re
        #Add user message to the conversation and assemble the prompt
        self.add_user_message(user_message)
        prompt = self.assemble_prompt(context, user_message)
        

        raw_response = complete_document_sdk(
            prefix=prompt,
            model=model,
            num_predict=max_tokens,
            temperature=temperature
        )
        
        #Cleaning the response
        if '</response>' in raw_response:
            raw_response = raw_response.split('</response>')[0]
        raw_response = re.sub(r'</?\w+>\s*$', '', raw_response)
        
        answer = raw_response

        #Adding the models response to the conversation
        self.add_assistant_message(answer)
        return answer
    

    #Print the conversation history
    def display_conversation(self):
        print("\n" + "="*50)
        print(f"Document Type: {self.document_type}")
        if self.system_prompt:
            print(f"System: {self.system_prompt}\n")
        for msg in self.history:
            role = "You" if msg['role'] == 'user' else "Assistant"
            print(f"{role}: {msg['content']}")
        print("="*50)

    def preview_prompt(self, context, question, print_output=True):
        prompt = self.assemble_prompt(context, question)
        if print_output:
            print("\n")
            print("Raw prompt:")
            print(prompt)
            print("---------------------------------")
            print(f"Approximate token count: {len(prompt.split())} words")

        
        return prompt



# Generation Configuration and chat loop

In [34]:
#Configuration variables
SYSTEM_PROMPT = "You are an HKBU research assistant. Answer short." #System prompt
CONTEXT = """""" # RAG Implementation will go here, might have to change it to be dynamic
DOCUMENT_TYPE = "structured"  # Options: 'structured', 'markdown', 'conversation'
MODEL = "gemma3:4b"  # Model configuration variable
TEMPERATURE = 0.3
MAX_TOKENS = 180
MAX_HISTORY_TURNS=3

#Create ConversationManager
conversation = ConversationManager(
    system_prompt=SYSTEM_PROMPT,
    document_type=DOCUMENT_TYPE,
    max_history_turns=MAX_HISTORY_TURNS
)

print("---------------------------------")
print("🤖 HKBU RESEARCH ASSISTANT")
print("---------------------------------")
print("Type 'exit' to quit")
print(f"System: {SYSTEM_PROMPT}")
print(f"Model: {MODEL}")
print("---------------------------------")

# Main conversation loop
user_input = ""
while user_input.lower() != 'exit':
    # Get user input
    user_input = input("\nType your message: ")
    
    if user_input.lower() == 'exit':
        print("\n👋 Goodbye!")
        break
    
    if not user_input.strip():
        continue
    
    
    print(f"\n👤 USER MESSAGE: {user_input}")
    
    #preview = conversation.preview_prompt(context, user_input) #Printing the prompt for debugging 
    # Get model response with configured parameters (conversation manager handles history and prompt assembly)
    response = conversation.get_response(
        user_message=user_input,
        context=CONTEXT,
        model=MODEL,  
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS
    )
    
    # Print assistant response 
    print(f"🤖 ASSISTANT RESPONSE: {response}")
    print("---------------------------------")

---------------------------------
🤖 HKBU RESEARCH ASSISTANT
---------------------------------
Type 'exit' to quit
System: You are an HKBU research assistant. Answer short.
Model: gemma3:4b
---------------------------------

👤 USER MESSAGE: HIII
🤖 ASSISTANT RESPONSE:  Hello.

---------------------------------

👤 USER MESSAGE: my name is bob
🤖 ASSISTANT RESPONSE:  Hello Bob.

---------------------------------

👤 USER MESSAGE: what is the weather today
🤖 ASSISTANT RESPONSE:  I cannot answer your question. I do not have access to real-time weather information.

---------------------------------

👤 USER MESSAGE: 1
🤖 ASSISTANT RESPONSE:  I cannot answer your question. I do not have access to real-time weather information.

---------------------------------

👤 USER MESSAGE: 2
🤖 ASSISTANT RESPONSE:  I cannot answer your question. I do not have access to real-time weather information.

---------------------------------

👤 USER MESSAGE: what is my name
🤖 ASSISTANT RESPONSE:  I do not know your n